In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

from models import *
use_gpu = torch.cuda.is_available()
batch_size = 128
model = VGG16_quant()

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

In [2]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))

trainloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))


testloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [3]:
# Verify Final Model Test Accuracy
fdir = f"final_model/final_model_checkpoints/"
checkpoint = torch.load(
    os.path.join(fdir, 'model_best.pth.tar'),
    map_location='cuda',
    weights_only=False
)
device = torch.device("cuda") 
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(),momentum=0.9)
model.load_state_dict(checkpoint['state_dict'])
optimizer.load_state_dict(checkpoint['optimizer'])

start_epoch = checkpoint['epoch']
best_prec = checkpoint['best_prec']
model.eval()
model.cuda()
test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)
print('\n Model Verified Test Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


 Model Verified Test Accuracy: 9254/10000 (93%)



In [4]:
print("Squeezed Layer 27:")
print(model.features[27])

Squeezed Layer 27:
QuantConv2d(
  8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)


In [5]:
model.eval()
w_bit = 4
layer = model.features[27]
weight_q = layer.weight_q 
w_alpha = layer.weight_quant.wgt_alpha.data.item()  
w_delta = w_alpha/(2**(w_bit-1)-1)   
weight_int = weight_q / w_delta 
print(weight_int[0]) 

tensor([[[ 7.0000,  7.0000, -7.0000],
         [ 7.0000,  7.0000,  7.0000],
         [ 7.0000, -7.0000,  7.0000]],

        [[-7.0000,  7.0000, -7.0000],
         [ 2.0000,  7.0000, -7.0000],
         [ 7.0000, -7.0000, -7.0000]],

        [[ 7.0000, -7.0000, -7.0000],
         [ 7.0000,  7.0000,  7.0000],
         [ 7.0000, -7.0000,  7.0000]],

        [[ 7.0000, -7.0000, -7.0000],
         [-7.0000, -7.0000, -7.0000],
         [-7.0000, -7.0000,  7.0000]],

        [[-7.0000, -5.0000, -7.0000],
         [-7.0000,  7.0000,  7.0000],
         [-7.0000,  7.0000,  7.0000]],

        [[-7.0000, -7.0000,  7.0000],
         [ 7.0000, -7.0000, -7.0000],
         [-7.0000, -7.0000,  7.0000]],

        [[ 7.0000, -7.0000,  7.0000],
         [ 7.0000,  7.0000,  7.0000],
         [ 3.0000, -7.0000,  7.0000]],

        [[-7.0000,  7.0000,  7.0000],
         [ 2.0000,  7.0000,  7.0000],
         [-7.0000,  7.0000, -7.0000]]], device='cuda:0',
       grad_fn=<SelectBackward0>)


In [6]:

input_batch, _ = next(iter(testloader))
input_batch = input_batch.cuda()

with torch.no_grad():
    model(input_batch)

In [7]:
x_bit = 4 
captured = {}
def pre_hook(module, inputs):
    captured['x'] = inputs[0].detach().clone()
    
def forward_hook(module, inputs, output):
    captured['y'] = output.detach().clone()
    
input_handle = layer.register_forward_pre_hook(pre_hook)
output_handle = layer.register_forward_hook(forward_hook)

with torch.no_grad():
    model(input_batch.cuda())

input_handle.remove()
output_handle.remove()

x = captured['x']
psum_ref = captured['y'] 

x_alpha  =  layer.act_alpha.item()
x_delta = x_alpha / (2**x_bit - 1)

act_quant_fn = act_quantization(x_bit) 
x_q = act_quant_fn(x, x_alpha)        

x_int = x_q / x_delta
print(x_int[0])

tensor([[[ 0.0000,  1.0000,  6.0000,  7.0000],
         [ 0.0000,  4.0000, 12.0000, 11.0000],
         [ 0.0000,  7.0000, 11.0000, 10.0000],
         [ 0.0000,  0.0000,  1.0000,  1.0000]],

        [[ 0.0000,  0.0000,  4.0000,  0.0000],
         [ 0.0000,  4.0000,  4.0000,  0.0000],
         [ 4.0000,  8.0000,  8.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000]],

        [[ 0.0000,  0.0000,  1.0000,  2.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  1.0000,  0.0000,  2.0000],
         [ 0.0000,  1.0000,  4.0000,  3.0000]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  1.0000,  1.0000]],

        [[ 0.0000,  2.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
   

In [8]:
#Verifying quant layer output matches manual (conv2d(x,w) * x_delta * w_delta)  (squeezed layer)
new_conv = nn.Conv2d(8, 8, kernel_size=3, stride=1, padding=1, bias=False).cuda()

with torch.no_grad():
    new_conv.weight.copy_(weight_int)
    psum_int = new_conv(x_int)

with torch.no_grad():
    psum_new = psum_int * x_delta * w_delta

with torch.no_grad():
    difference = (psum_new - psum_ref).abs()
print("Verifying quant layer output matches manual conv(x,w) * deltax * deltaw")
print("Mean absolute error compared to original output:", difference.mean().item())
print("Maximum absolute error:", difference.max().item())

Verifying quant layer output matches manual conv(x,w) * deltax * deltaw
Mean absolute error compared to original output: 9.340296713844509e-08
Maximum absolute error: 1.9073486328125e-06


In [9]:
#Verifying quant layer (squeezed layer) output after ReLU matches next layer's input

captured_next = {}

def next_pre_hook(module, inputs):
    captured_next["x"] = inputs[0].detach().clone()

handle = model.features[29].register_forward_pre_hook(next_pre_hook)

with torch.no_grad():
    model(input_batch.cuda())

handle.remove()

psum_recovered = torch.relu(psum_int * x_delta * w_delta)

difference = (psum_recovered - captured_next["x"]).abs()

print("Verifying quant layer (squeezed layer) output after ReLU matches next layer's input")
print("Mean absolute error:", difference.mean().item())
print("Maximum absolute error:", difference.max().item())


Verifying quant layer (squeezed layer) output after ReLU matches next layer's input
Mean absolute error: 6.561631948898139e-08
Maximum absolute error: 1.9073486328125e-06


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def verify_psum_recovered(model, testloader, num_images=1):

    # --------------------------------------------------
    # 1. Identify the squeezed layer and following layer
    # --------------------------------------------------
    conv = model.features[27]
    next_conv = model.features[29]

    assert conv.in_channels == 8
    assert conv.out_channels == 8
    assert conv.bit == 4
    assert conv.bias is None

    assert isinstance(model.features[28], nn.ReLU)
    assert isinstance(next_conv, nn.Conv2d)
    assert next_conv.in_channels == 8

    # --------------------------------------------------
    # 2. Register forward prehooks
    # --------------------------------------------------
    stored = {}

    def save_conv_input(module, inputs):
        stored["conv_input"] = inputs[0].detach().clone()

    def save_next_input(module, inputs):
        stored["prehook_next"] = inputs[0].detach().clone()

    hook1 = conv.register_forward_pre_hook(save_conv_input)
    hook2 = next_conv.register_forward_pre_hook(save_next_input)

    was_training = model.training
    original_tf32 = torch.backends.cudnn.allow_tf32

    try:
        model.eval()

        # Disable TF32 for a more accurate FP32 reference
        torch.backends.cudnn.allow_tf32 = False

        # --------------------------------------------------
        # 3. Execute forward pass on test images
        # --------------------------------------------------
        images, _ = next(iter(testloader))

        device = next(model.parameters()).device
        images = images[:num_images].to(device)

        with torch.no_grad():
            _ = model(images)

            # --------------------------------------------------
            # 4. Obtain actual quantized activations and weights
            # --------------------------------------------------

            # Quantize prehooked input using the layer's quantizer
            x_q = conv.act_alq(
                stored["conv_input"],
                conv.act_alpha
            ).detach()

            # Quantized weights saved during the forward pass
            w_q = conv.weight_q.detach().clone()

            # --------------------------------------------------
            # 5. Calculate quantization step sizes
            # --------------------------------------------------

            # 4-bit unsigned activations: 0 to 15
            delta_x = (
                conv.act_alpha.detach().double()
                / (2**conv.bit - 1)
            )

            # 4-bit signed weights: -7 to 7
            delta_w = (
                conv.weight_quant.wgt_alpha.detach().double()
                / (2**(conv.bit - 1) - 1)
            )

            # --------------------------------------------------
            # 6. Recover integer activations and weights
            # --------------------------------------------------

            x_int = torch.round(
                x_q.double() / delta_x
            ).to(torch.int64)

            w_int = torch.round(
                w_q.double() / delta_w
            ).to(torch.int64)

            # Verify integer ranges
            assert x_int.min().item() >= 0
            assert x_int.max().item() <= 15

            assert w_int.min().item() >= -7
            assert w_int.max().item() <= 7

            # --------------------------------------------------
            # 7. Perform integer-valued convolution
            # --------------------------------------------------

            # FP64 represents these small integers exactly.
            # Integer products and sums are therefore exact
            # for this convolution's accumulation range.

            psum_int = F.conv2d(
                x_int.double(),
                w_int.double(),
                bias=None,
                stride=conv.stride,
                padding=conv.padding,
                dilation=conv.dilation,
                groups=conv.groups
            )

            # --------------------------------------------------
            # 8. Recover floating-point psum and apply ReLU
            # --------------------------------------------------

            psum_recovered = F.relu(
                psum_int * delta_x * delta_w
            )

            # --------------------------------------------------
            # 9. Get the next layer's prehooked input
            # --------------------------------------------------

            prehook_next = stored["prehook_next"].double()

            assert psum_recovered.shape == prehook_next.shape

            # --------------------------------------------------
            # 10. Compare recovered psum with hooked reference
            # --------------------------------------------------

            abs_error = (
                psum_recovered - prehook_next
            ).abs()

            max_error = abs_error.max().item()
            mean_error = abs_error.mean().item()

            # --------------------------------------------------
            # 11. Display verification results
            # --------------------------------------------------

            print("=== Layer Information ===")
            print("Squeezed layer:", conv)
            print("Input shape:", tuple(x_int.shape))
            print("Weight shape:", tuple(w_int.shape))
            print("Output shape:", tuple(psum_recovered.shape))

            print("\n=== Quantization Parameters ===")
            print("delta_x:", delta_x.item())
            print("delta_w:", delta_w.item())

            print("\n=== Integer Ranges ===")
            print("x_int min/max:",
                  x_int.min().item(),
                  x_int.max().item())

            print("w_int min/max:",
                  w_int.min().item(),
                  w_int.max().item())

            print("\n=== Verification Results ===")
            print("Maximum absolute error:", max_error)
            print("Mean absolute error:", mean_error)

            print("Ideal threshold (< 1e-5):",
                  max_error < 1e-5)

            print("Project threshold (< 1e-3):",
                  max_error < 1e-3)

            print("\n=== Sample Outputs ===")
            print("psum_recovered:",
                  psum_recovered.flatten()[:8].tolist())

            print("prehook_next:",
                  prehook_next.flatten()[:8].tolist())

            if max_error < 1e-3:
                print("\nPASS: psum_recovered matches prehook")
            else:
                print("\nFAIL: Error exceeds project threshold")

            return {
                "x_int": x_int,
                "w_int": w_int,
                "psum_int": psum_int,
                "psum_recovered": psum_recovered,
                "prehook_next": prehook_next,
                "max_error": max_error,
                "mean_error": mean_error,
                "delta_x": delta_x,
                "delta_w": delta_w
            }

    finally:
        # Remove hooks and restore original model settings
        hook1.remove()
        hook2.remove()

        torch.backends.cudnn.allow_tf32 = original_tf32
        model.train(was_training)


# Run verification on the existing trained model
results = verify_psum_recovered(
    model,
    testloader,
    num_images=4
)

=== Layer Information ===
Squeezed layer: QuantConv2d(
  8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)
Input shape: (4, 8, 4, 4)
Weight shape: (8, 8, 3, 3)
Output shape: (4, 8, 4, 4)

=== Quantization Parameters ===
delta_x: 0.07419142723083497
delta_w: 0.09907335042953491

=== Integer Ranges ===
x_int min/max: 0 15
w_int min/max: -7 7

=== Verification Results ===
Maximum absolute error: 8.352205611572572e-07
Mean absolute error: 5.9570513223290066e-08
Ideal threshold (< 1e-5): True
Project threshold (< 1e-3): True

=== Sample Outputs ===
psum_recovered: [0.051452752882354964, 0.30871651729412974, 0.9776023047647443, 0.9702519114958364, 0.0, 0.0, 1.8449487104958708, 2.116913261445461]
prehook_next: [0.05145272985100746, 0.3087165355682373, 0.9776023626327515, 0.9702519178390503, 0.0, 0.0, 1.8449488878250122, 2.116913080215454]

PASS: psum_recovered matches prehook
